# Exercise: random numbers with `numpy.random`

## Introduction

`numpy.random` generates arrays of (pseudo-)random numbers following many probability distributions.
This is useful to simulate noisy data, test an analysis on data whose ground truth is known, or run Monte-Carlo simulations.

The steps are:

1. Discover the generic principles: the random generator, the seed, the `size` argument
2. Play with the 3 most common distributions: uniform, gaussian (normal) and Poisson
3. Simulate the Poisson noise of a detector counting photons

Useful documentation: https://numpy.org/doc/stable/reference/random/index.html

## Part I: generic principles

In [ ]:
import numpy as np

In [ ]:
%matplotlib widget
# This requires ipympl
# Or for non-interactive plots: %matplotlib inline

from matplotlib import pyplot as plt

### The random generator

Random numbers are drawn from a `Generator` object, created with `np.random.default_rng()`.

Create a generator and draw 5 random floats in $[0, 1)$ (hint: `rng.random`). Run the cell several times: what happens?

In [ ]:
rng = # TODO
values = # TODO
print(values)

Note: you may encounter older code using functions like `np.random.rand`, `np.random.randn` or `np.random.seed`.
This *legacy* API is still available, but the `Generator` API is recommended for new code.

### Reproducibility: the seed

The numbers are not truly random: they are computed by an algorithm starting from an initial state, the **seed**.
By default, the seed is taken from the operating system, so each run is different.

Create two generators with the same seed (hint: `np.random.default_rng(seed=42)`) and check that they produce the same numbers.

In [ ]:
rng1 = # TODO
rng2 = # TODO
print(rng1.random(5))
print(rng2.random(5))

Fixing the seed makes a simulation (or a unit test) reproducible: use it whenever you need to get the same "random" data again.

### The `size` argument

All the drawing methods take a `size` argument, which is the shape of the returned array.

Draw a `(3, 4)` array of random floats.

In [ ]:
rng = np.random.default_rng(seed=0)
array = # TODO
print(array)

### Integers and choices

Simulate 10000 rolls of a die (hint: `rng.integers`, beware: the `high` bound is excluded).

Count how many times each face came out (hint: `np.unique(..., return_counts=True)` or `np.bincount`).

In [ ]:
rolls = # TODO
faces, face_counts = # TODO
print(faces, face_counts)

In [ ]:
fig = plt.figure()
plt.bar(faces, face_counts)
plt.xlabel("Face")
plt.ylabel("Count")
plt.title("10000 rolls of a die")

The generator can also pick elements of an array (`rng.choice`) or shuffle them (`rng.permutation`).

* Pick 3 different names out of the `trainees` list
* Shuffle the list

In [ ]:
trainees = np.array(["Alice", "Bob", "Charlie", "Dan", "Eve", "Frank"])
# TODO

## Part II: three distributions

### Uniform distribution

`rng.uniform(low, high, size)` draws floats uniformly distributed in $[low, high)$.

Draw 100000 values between -1 and 3, and compare their mean and standard deviation with the theoretical values $\frac{low + high}{2}$ and $\frac{high - low}{\sqrt{12}}$.

In [ ]:
uniform = # TODO
expected_mean = # TODO
expected_std = # TODO
print("mean:", np.mean(uniform), "expected:", expected_mean)
print("std:", np.std(uniform), "expected:", expected_std)

In [ ]:
fig = plt.figure()
plt.hist(uniform, bins=100)
plt.title("Uniform distribution")

#### Monte-Carlo estimation of $\pi$

Draw N random points uniformly in the square $[-1, 1] \times [-1, 1]$.
The fraction of points falling inside the unit circle ($x^2 + y^2 < 1$) is the ratio of the areas: $\frac{\pi}{4}$.

Estimate $\pi$ with N = 100, 10000 and 1000000 points. How does the precision evolve?

In [ ]:
for n in (100, 10_000, 1_000_000):
    points = # TODO
    pi_estimate = # TODO
    print(n, pi_estimate)

### Gaussian (normal) distribution

`rng.normal(loc, scale, size)` draws values from a gaussian distribution of mean `loc` and standard deviation `scale`:

$$
p(x) = \frac{1}{\sigma \sqrt{2 \pi}} \exp\left(-\frac{(x - \mu)^2}{2 \sigma^2}\right)
$$

Draw 100000 values with $\mu = 10$ and $\sigma = 2$, and compute the theoretical density `pdf` at each `x` position.

In [ ]:
mu, sigma = 10, 2
gaussian = # TODO

x = np.linspace(0, 20, 200)
pdf = # TODO

In [ ]:
fig = plt.figure()
plt.hist(gaussian, bins=100, density=True, label="histogram (density=True)")
plt.plot(x, pdf, label="theoretical density")
plt.legend()
plt.title("Gaussian distribution")

Compute the fraction of values within $\mu \pm \sigma$, $\mu \pm 2\sigma$ and $\mu \pm 3\sigma$ (expected: 68.3%, 95.4% and 99.7%).

In [ ]:
for k in (1, 2, 3):
    fraction = # TODO
    print(f"within {k} sigma: {100 * fraction:.1f}%")

### Poisson distribution

The Poisson distribution gives the number of events (e.g. photons hitting a pixel) occurring during a given time, when they arrive independently at a constant average rate.
With $\lambda$ the mean number of events, the probability to count $k$ events is:

$$
P(k) = \frac{\lambda^k e^{-\lambda}}{k!}
$$

The values are **integers**, and the **variance is equal to the mean**: $\sigma = \sqrt{\lambda}$.

Draw 100000 values with `rng.poisson` for $\lambda = 3$, and check that their mean and variance are both close to 3.

In [ ]:
lam = 3
counts = # TODO
print("dtype:", counts.dtype, "min:", counts.min(), "max:", counts.max())
print("mean:", np.mean(counts), "variance:", np.var(counts))

Compute the frequency of each value $k$ (hint: `np.bincount`, divided by the number of values) and compare with $P(k)$ (hint: `scipy.special.factorial`, or `math.factorial` in a loop).

In [ ]:
frequencies = # TODO
k = np.arange(len(frequencies))
probabilities = # TODO

In [ ]:
fig = plt.figure()
plt.bar(k, frequencies, label="drawn")
plt.plot(k, probabilities, "ro", label="P(k)")
plt.xlabel("k")
plt.legend()
plt.title(f"Poisson distribution, lambda={lam}")

For large $\lambda$, the Poisson distribution gets close to a gaussian of mean $\lambda$ and standard deviation $\sqrt{\lambda}$.

Compare the histograms of 100000 draws of a Poisson distribution with $\lambda = 100$ and of a gaussian distribution with $\mu = 100$ and $\sigma = 10$.

In [ ]:
poisson_100 = # TODO
gaussian_100 = # TODO

In [ ]:
fig = plt.figure()
bins = np.arange(50, 151)
plt.hist(poisson_100, bins=bins, alpha=0.5, label="Poisson, lambda=100")
plt.hist(gaussian_100, bins=bins, alpha=0.5, label="gaussian, mu=100, sigma=10")
plt.legend()

## Part III: Poisson noise on photon counts

A photon-counting detector records an image of a diffraction spot on top of a flat background.
The following cell computes the **expected** number of photons in each pixel, for a given exposure: this is the noise-free image.

In [ ]:
y, x = np.mgrid[0:100, 0:100]
spot = np.exp(-((x - 50) ** 2 + (y - 50) ** 2) / (2 * 10 ** 2))

def expected_photons(exposure):
    """Expected photon counts per pixel for an exposure time in seconds"""
    return exposure * (20 * spot + 2)  # in photons per pixel

### Adding Poisson noise

Each pixel counts a random number of photons following a Poisson distribution whose mean is the expected count of that pixel.

`rng.poisson` accepts an **array** of $\lambda$: it draws one value per element of the array, each with its own mean.

Simulate the measured image for an exposure of 1 s.

In [ ]:
rng = np.random.default_rng(seed=0)
expected = expected_photons(1)
measured = # TODO

In [ ]:
fig, axes = plt.subplots(1, 2, sharex=True, sharey=True)
axes[0].imshow(expected)
axes[0].set_title("Expected")
axes[1].imshow(measured)
axes[1].set_title("Measured")

Note: "adding" Poisson noise does **not** mean adding something to the data!
The common mistakes are:

* `data + rng.poisson(data)`: it doubles the signal
* `data + rng.normal(scale=sigma, size=data.shape)` with the same `sigma` everywhere: the Poisson noise depends on the signal
* applying `rng.poisson` to normalized data (e.g. values between 0 and 1): the data must be expressed in number of photons

If needed, the noise itself is `measured - expected`.

### Noise versus exposure

Simulate the measured image for exposures of 0.1 s, 1 s, 10 s and 100 s, and plot the profile along the row 50 of each measured image, divided by the exposure, and the expected profile for 1 s.

In [ ]:
fig = plt.figure()
for exposure in (0.1, 1, 10, 100):
    measured = # TODO
    plt.plot(measured[50] / exposure, label=f"{exposure} s")
plt.plot(expected_photons(1)[50], "k", label="expected")
plt.xlabel("x")
plt.ylabel("Photons per second")
plt.legend()

The absolute noise ($\sqrt{N}$) increases with the number of photons $N$, but the **relative noise** $\frac{\sqrt{N}}{N} = \frac{1}{\sqrt{N}}$ decreases: the signal-to-noise ratio is $\sqrt{N}$.
To halve the relative noise, the exposure must be multiplied by 4.

### Variance equals mean

Simulate 1000 acquisitions of 1 s at once (hint: with `size`, the shape of `lam` must be at the end of the shape of the result).
For each pixel, compute the mean and the variance over the 1000 acquisitions, and plot the variance as a function of the mean.

In [ ]:
acquisitions = # TODO
pixel_mean = # TODO
pixel_variance = # TODO
print(acquisitions.shape)

In [ ]:
fig = plt.figure()
plt.plot(pixel_mean.ravel(), pixel_variance.ravel(), ".", alpha=0.2)
plt.plot([0, 22], [0, 22], "k", label="variance = mean")
plt.xlabel("Mean over acquisitions")
plt.ylabel("Variance over acquisitions")
plt.legend()

### Bonus: dead time

Real detectors miss photons arriving too close to each other.
With probability 0.9 of detecting each photon, the detected counts are drawn from a binomial distribution (`rng.binomial(n, p)`) where `n` is the number of photons that hit the pixel.

Simulate it for the 1000 acquisitions above. Is the variance still equal to the mean?
(Answer: yes! Randomly losing Poisson events gives a Poisson distribution of mean $0.9 \lambda$)

In [ ]:
detected = # TODO
ratio = np.var(detected, axis=0) / np.mean(detected, axis=0)
print("variance / mean, averaged over pixels:", np.mean(ratio))

## Solution

<details><summary>...</summary>


```python
# Part I

import numpy as np

# The random generator
rng = np.random.default_rng()
values = rng.random(5)  # different at each run

# Reproducibility: the seed
rng1 = np.random.default_rng(seed=42)
rng2 = np.random.default_rng(seed=42)

# The size argument
array = rng.random(size=(3, 4))

# Integers and choices
rolls = rng.integers(1, 7, size=10000)
faces, face_counts = np.unique(rolls, return_counts=True)
# or: face_counts = np.bincount(rolls)[1:]

print(rng.choice(trainees, size=3, replace=False))
print(rng.permutation(trainees))

# Part II

# Uniform distribution
uniform = rng.uniform(-1, 3, size=100000)
expected_mean = (-1 + 3) / 2
expected_std = (3 - -1) / np.sqrt(12)

# Monte-Carlo estimation of pi
for n in (100, 10_000, 1_000_000):
    points = rng.uniform(-1, 1, size=(2, n))
    pi_estimate = 4 * np.mean(points[0] ** 2 + points[1] ** 2 < 1)
    print(n, pi_estimate)
# The error decreases as 1 / sqrt(n): 100 times more points for 10 times more precision

# Gaussian distribution
gaussian = rng.normal(mu, sigma, size=100000)
pdf = np.exp(-(x - mu) ** 2 / (2 * sigma ** 2)) / (sigma * np.sqrt(2 * np.pi))

for k in (1, 2, 3):
    fraction = np.mean(np.abs(gaussian - mu) < k * sigma)
    print(f"within {k} sigma: {100 * fraction:.1f}%")

# Poisson distribution
counts = rng.poisson(lam, size=100000)  # integers

frequencies = np.bincount(counts) / counts.size
k = np.arange(len(frequencies))
from scipy.special import factorial
probabilities = lam ** k * np.exp(-lam) / factorial(k)
# or without scipy:
# import math
# probabilities = lam ** k * np.exp(-lam) / np.array([math.factorial(i) for i in k])

poisson_100 = rng.poisson(100, size=100000)
gaussian_100 = rng.normal(100, 10, size=100000)

# Part III

# Adding Poisson noise
measured = rng.poisson(expected)

# Noise versus exposure
for exposure in (0.1, 1, 10, 100):
    measured = rng.poisson(expected_photons(exposure))

# Variance equals mean
acquisitions = rng.poisson(expected, size=(1000,) + expected.shape)
pixel_mean = np.mean(acquisitions, axis=0)
pixel_variance = np.var(acquisitions, axis=0)

# Bonus: dead time
detected = rng.binomial(acquisitions, 0.9)
```

</details>